# Forecast workbench

**Purpose**: explore the curated features, tweak the forecasting model's levers, and
visualize the resulting season forecast, all against the live database.

**Exception to `notebooks/README.md`'s one-question-per-notebook convention**: this
notebook has three sections instead of one conclusion, and is meant to be revisited
repeatedly rather than deleted once it reaches an answer. It is closer in spirit to
design.md section 10.2's "local exploration surface" than to a single-conclusion EDA
notebook -- see the plan this was built from and ADR 0003 for the reasoning. All chart
logic still lives in `src/plforecast/viz.py`, tested and importable, per ADR 0003's
"notebooks consume `plforecast`, never define logic": nothing below defines a metric,
a model, or a chart shape that isn't already in `src/plforecast`.

1. **Feature EDA** -- ad hoc looks at the curated tables the model trains on.
2. **Lever workbench** -- change `MODEL`/`XI`/`BLEND`/`RHO`/etc. and re-run to see the
   backtest respond.
3. **Forecast visualization** -- a live forecast under the current levers, plus the
   longer-run view from committed artifacts and evaluation reports.


In [ ]:
from pathlib import Path

import plotly.express as px
import polars as pl

from plforecast import forecasting, viz
from plforecast.config import Settings
from plforecast.evaluate.backtest import run_backtest
from plforecast.evaluate.calibration import calibration_by_outcome
from plforecast.evaluate.metrics import outcome_index
from plforecast.evaluate.report import format_table
from plforecast.evaluate.tuning import tune_parameter
from plforecast.features.schedule import build_rest_days
from plforecast.features.strength import build_club_strength
from plforecast.models.base import drop_odds_columns
from plforecast.storage.db import connect

# Jupyter's working directory is this notebook's own directory, not the repo root
# (story A-14's config injection makes this a one-line fix rather than a chdir hack).
REPO_ROOT = Path("../..")
NOTEBOOK_CONFIG = Settings(data_dir=REPO_ROOT / "data")
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
EVALUATION_DIR = REPO_ROOT / "docs" / "evaluation"

## Section 1: Feature EDA

Reads `stg_matches`, `mart_team_match`, and `stg_fixtures` straight from the curated
database. Nothing here is reused elsewhere -- it's ad hoc `plotly.express`, not a
tested chart -- so it stays inline rather than moving into `viz.py`.


In [ ]:
conn = connect(NOTEBOOK_CONFIG)
matches = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result FROM stg_matches ORDER BY date"
).pl()
team_match = conn.execute("SELECT * FROM mart_team_match").pl()
fixtures = conn.execute("SELECT * FROM stg_fixtures ORDER BY kickoff_time").pl()
conn.close()

matches.height, team_match.height, fixtures.height

### xG vs. goals

How much noise sits between a club's underlying chances and the scoreline.

In [ ]:
xg_goals = team_match.drop_nulls(["xg_for"])
fig = px.scatter(
    xg_goals.to_pandas(),
    x="xg_for",
    y="goals_for",
    color="is_home",
    opacity=0.4,
    title="Goals for vs. xG for, per club-match",
)
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=xg_goals["xg_for"].max(),
    y1=xg_goals["xg_for"].max(),
    line={"dash": "dash", "color": "gray"},
)
fig.show()

### Home-advantage drift over time

Mean home goal difference by season -- is home advantage stable, shrinking, or noisy?

In [ ]:
home_advantage = (
    matches.with_columns((pl.col("home_goals") - pl.col("away_goals")).alias("home_gd"))
    .group_by("season")
    .agg(pl.col("home_gd").mean().alias("mean_home_gd"), pl.len().alias("matches"))
    .sort("season")
)
px.line(
    home_advantage.to_pandas(),
    x="season",
    y="mean_home_gd",
    markers=True,
    title="Mean home goal difference by season",
)

### Rest-day distribution

`features/schedule.py`'s `build_rest_days` on the current season's fixture list.

In [ ]:
rest_days = build_rest_days(fixtures)
rest_long = pl.concat(
    [
        rest_days.select(pl.col("home_rest_days").alias("rest_days")).with_columns(
            pl.lit("home").alias("venue")
        ),
        rest_days.select(pl.col("away_rest_days").alias("rest_days")).with_columns(
            pl.lit("away").alias("venue")
        ),
    ]
).drop_nulls("rest_days")
px.histogram(
    rest_long.to_pandas(),
    x="rest_days",
    color="venue",
    barmode="overlay",
    nbins=20,
    title="Rest days between fixtures, current season",
)

### Club strength trend

`features/strength.py`'s `build_club_strength` at a handful of `as_of` dates, for a
couple of clubs -- change `CLUBS_TO_TRACK` to look at others. Uses whichever club IDs
actually appear in `stg_matches`; the first two alphabetically are a reasonable
default.

In [ ]:
CLUBS_TO_TRACK = sorted(matches["home_club_id"].unique().to_list())[:2]
AS_OF_DATES = [d for d in matches["date"].unique().sort().to_list()[::40]]

strength_rows = []
for as_of in AS_OF_DATES:
    strength = build_club_strength(matches, as_of=as_of, xi=0.0018)
    for row in strength.filter(pl.col("club_id").is_in(CLUBS_TO_TRACK)).iter_rows(named=True):
        strength_rows.append({"as_of": as_of, **row})

strength_trend = pl.DataFrame(strength_rows)
px.line(
    strength_trend.to_pandas(),
    x="as_of",
    y="home_attack_rate",
    color="club_id",
    markers=True,
    title="Home attack rate over time (decayed, xi=0.0018)",
)

## Section 2: Lever workbench

Change the parameters below and re-run this section to see the backtest respond. These
mirror `plforecast evaluate`/`plforecast tune`'s own levers -- nothing here is a new
metric or a new model, only different settings for the existing ones.


In [ ]:
# ---- parameters: edit and re-run this cell and everything below it ----
MODEL = "dixon-coles"  # "poisson" | "dixon-coles" | "xg-rates"
XI = None  # None = use config.py's tuned default for MODEL
BLEND = None  # xg-rates only
RHO = None  # xg-rates only
MIN_TRAIN_MATCHES = 100
CADENCE = "date"  # "date" | "gameweek"

In [ ]:
def make_factory(name, *, xi=XI, blend=BLEND, rho=RHO):
    return forecasting.build_model_factory(name, xi=xi, blend=blend, rho=rho)


conn = connect(NOTEBOOK_CONFIG)
evaluation_matches = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result, benchmark_home_odds, benchmark_draw_odds, "
    "benchmark_away_odds, benchmark_source "
    "FROM stg_matches WHERE season != (SELECT max(season) FROM stg_matches) ORDER BY date"
).pl()
conn.close()

backtest_result = run_backtest(
    evaluation_matches,
    make_factory(MODEL),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
scores = backtest_result.scores
print(f"scored: {backtest_result.n_scored}, unrateable: {backtest_result.unrateable.height}")
print(
    format_table(
        [
            {
                "model": MODEL,
                "n": scores.height,
                "rps": scores["rps"].mean(),
                "log_loss": scores["log_loss"].mean(),
                "brier": scores["brier"].mean(),
            }
        ]
    )
)

Calibration under the current levers:

In [ ]:
probs = scores.select("p_home", "p_draw", "p_away").to_numpy()
outcomes = outcome_index(scores["result"].to_list())
calibration_rows = calibration_by_outcome(probs, outcomes).to_dicts()
viz.calibration_chart(calibration_rows)

Scoreline matrices for a couple of illustrative fixtures, fit on the full evaluation
window under the current levers -- the most direct "change a lever, see the model
change" visual.

In [ ]:
illustrative_fitted = make_factory(MODEL)().fit(drop_odds_columns(evaluation_matches))
illustrative_fixtures = (
    evaluation_matches.filter(pl.col("season") == evaluation_matches["season"].max())
    .select("home_club_id", "away_club_id")
    .unique()
    .head(2)
)
for home_id, away_id in illustrative_fixtures.iter_rows():
    heatmap = viz.scoreline_heatmap(
        illustrative_fitted.scoreline_matrix(home_id, away_id, max_goals=6),
        home_id,
        away_id,
    )
    heatmap.show()

### Sweep one lever

Sweeps `xi` over a small grid via `evaluate.tuning.tune_parameter` (the same function
`plforecast tune` uses) and plots selection vs. report RPS. Swap `PARAMETER`/`GRID` to
sweep something else.

In [ ]:
PARAMETER = "xi"
GRID = [0.0005, 0.001, 0.0015, 0.002, 0.0025, 0.003]
SELECT_THROUGH_SEASON = sorted(evaluation_matches["season"].unique().to_list())[-3]

tuning_result = tune_parameter(
    evaluation_matches,
    lambda value: make_factory(MODEL, xi=value),
    GRID,
    parameter=PARAMETER,
    select_through_season=SELECT_THROUGH_SEASON,
    min_train_matches=MIN_TRAIN_MATCHES,
)
print(f"selected {PARAMETER} = {tuning_result['selected']}")
viz.lever_comparison(tuning_result["grid"], metric="rps", x_key="value")

## Section 3: Forecast visualization

A live forecast under section 2's levers via `forecasting.fit_and_simulate` (the same
function `plforecast forecast` calls), then the longer-run view from committed
artifacts and evaluation reports.


In [ ]:
run = forecasting.fit_and_simulate(
    MODEL, xi=XI, blend=BLEND, rho=RHO, simulations=5_000, seed=42, config=NOTEBOOK_CONFIG
)
print(
    f"{run.model_name}, season {run.season_label}, as of gameweek {run.as_of_gameweek}, "
    f"promoted-club prior applied to: {run.promoted_clubs or 'none'}"
)

Position probability matrix and projected points distribution from this live simulation:

In [ ]:
viz.position_matrix(run.result.club_ids, run.result.position_pmf, run.display_names)

In [ ]:
viz.points_ridgeline(run.result.club_ids, run.result.points, run.display_names)

Forecast evolution across every committed, gameweek-stamped forecast for the current
season (empty until a second gameweek's forecast has been committed) -- change
`CLUB_TO_TRACK` to look at another club.

In [ ]:
CLUB_TO_TRACK = run.result.club_ids[0]
viz.forecast_evolution(ARTIFACTS_DIR, run.season_label.replace("/", "-"), CLUB_TO_TRACK)

Calibration from the last committed evaluation report (`just evaluate`), for the
shipped model -- the longer-run view, distinct from section 2's live re-run above.

In [ ]:
import json

metrics_path = EVALUATION_DIR / "metrics.json"
if metrics_path.exists():
    report = json.loads(metrics_path.read_text())
    shipped = report["shipped_model"]
    viz.calibration_chart(report["calibration"].get(shipped, []))
else:
    print(f"no evaluation report yet at {metrics_path}; run `just evaluate` first")